
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>


# Demo - Model Testing Job with the Databricks CLI

This Demo guides you through creating a Lakeflow Jobs using the Databricks CLI. The Job will include tasks for model tracking with MLflow and testing the model. We will configure the Job using a JSON configuration file and submit it via Databricks CLI.

**Learning Objectives**

By the end of this demo, you will be able to:
- **Install the Databricks CLI** and **configure authentication** for a Databricks workspace, building upon credentials set in a previous notebook.
- **Execute the `help` command** within the Databricks CLI to explore available functionalities.
- **Manage and execute Databricks Job commands** directly from the notebook.


## REQUIRED - SELECT CLASSIC COMPUTE
Before executing cells in this notebook, please select your classic compute cluster in the lab. Be aware that **Serverless** is enabled by default.
Follow these steps to select the classic compute cluster:
1. Navigate to the top-right of this notebook and click the drop-down menu to select your cluster. By default, the notebook will use **Serverless**.
1. If your cluster is available, select it and continue to the next cell. If the cluster is not shown:
   - In the drop-down, select **More**.
   - In the **Attach to an existing compute resource** pop-up, select the first drop-down. You will see a unique cluster name in that drop-down. Please select that cluster.

**NOTE:** If your cluster has terminated, you might need to restart it in order to select it. To do this:
1. Right-click on **Compute** in the left navigation pane and select *Open in new tab*.
1. Find the triangle icon to the right of your compute cluster name and click it.
1. Wait a few minutes for the cluster to start.
1. Once the cluster is running, complete the steps above to select your cluster.

## Requirements
Please review the following requirements before starting the lesson:
- To run this notebook, you need to use one of the following Databricks runtime(s): **`17.3.x-cpu-ml-scala2.13`**

## Classroom Setup

Before starting the demo, run the provided classroom setup script. This script will define configuration variables necessary for the demo.

**Important:** Ensure that you have completed the [**`0 - Generate Tokens`**]($./0 - Generate Tokens) notebook beforehand. This will set up the necessary authentication credentials.

**If you have not done so, complete the [**`0 - Generate Tokens`**]($./0 - Generate Tokens) notebook now, then start this notebook again from the beginning to ensure proper setup.**

Execute the following cell:


In [0]:
%run ../Includes/Classroom-Setup-02

**Other Conventions:**

Throughout this demo, we'll refer to the object `DA`. This object, provided by Databricks Academy, contains variables such as your username, catalog name, schema name, working directory, and dataset locations. Run the code block below to view these details:

In [0]:
print(f"Username:          {DA.username}")
print(f"Catalog Name:      {DA.catalog_name}")
print(f"Schema Name:       {DA.schema_name}")
print(f"Working Directory: {DA.paths.working_dir}")
print(f"Dataset Location:  {DA.paths.datasets}")

### Authentication

Usually, you would have to set up authentication for the CLI. But in this training environment, that's already taken care of if you ran through the accompanying 
**'Generate Tokens'** notebook. 
If you did, credentials will already be loaded into the **`DATABRICKS_HOST`** and **`DATABRICKS_TOKEN`** environment variables. 

#####*If you did not, run through it now then restart this notebook.*

In [0]:
DA.get_credentials()


### Install CLI

Install the Databricks CLI using the following cell. Note that this procedure removes any existing version that may already be installed, and installs the newest version of the [Databricks CLI](https://docs.databricks.com/en/dev-tools/cli/index.html). A legacy version exists that is distributed through **`pip`**, however we recommend following the procedure here to install the newer one.

In [0]:
%sh rm -f $(which databricks); curl -fsSL https://raw.githubusercontent.com/databricks/setup-cli/v0.211.0/install.sh | sh

### Notebook Path Setup Continued
This code cell performs the following setup tasks:

- Retrieves the current Databricks cluster ID and displays it.
- Identifies the path of the currently running notebook.
- Constructs paths to related notebooks for Training and deploying the model, Performance Testing,  Model Prediction Analysis, and printing the Summary report of the Model testing. These paths are printed to confirm their accuracy.

In [0]:
# Retrieve the current cluster ID
cluster_id = spark.conf.get("spark.databricks.clusterUsageTags.clusterId")
print(cluster_id)

# Get the current notebook path
notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()

# Base path for related notebooks
base_path = notebook_path.rsplit('/', 1)[0] + "/2.2 Demo Pipeline - Testing Model Performance"

# Paths for specific process notebooks
Deploy_model_notebook_path = f"{base_path}/2.2a Train and Deploy Model"
print(Deploy_model_notebook_path)
Parallel_Task_1_notebook_path = f"{base_path}/2.2b Parallel Task 1 - Performance Test" 
print(Parallel_Task_1_notebook_path)
Parallel_Task_2_notebook_path = f"{base_path}/2.2b Parallel Task 2 - Model Prediction Analysis" 
print(Parallel_Task_2_notebook_path)
Summary_report_path = f"{base_path}/2.2c Model Summary Report"
print(Summary_report_path)

## Using the CLI
The cell below illustrates simple CLI usage. This displays a list of the contents of the /Users workspace directory.

In [0]:
%sh databricks workspace list /Users/

The CLI provides access to a broad swath of Databricks functionality. Use the **`help`** command to access the online help.

In [0]:
%sh databricks help

In [0]:
%sh databricks jobs list

## Configuration of a Lakeflow Job

This code cell constructs a JSON configuration string for a Databricks Lakeflow job. The configuration specifies a series of tasks designed to handle the deployment and inference stages for a model called "churn-prediction". Here are the key components and their functions:

**General Settings:**
> - **Job Naming**: The job is named using the current user's username with the suffix `-testing-workflow-job`.
> - **Concurrent Runs**: The job runs sequentially with a maximum of one concurrent run.
> - **Email Notifications**: Configured to alert the user on failure.
> - **Notification Settings**: Ensures alerts are not skipped for canceled or skipped runs.
> - **Timeout**: No timeout is set for the job.
> - **Queue Enabled**: Ensures tasks are queued if the job is already running.

**Tasks Defined:**
> 1. **Deploy Trained Model**: Deploys the trained model.
> 2. **Model Performance Test**: Tests the model's performance.
> 3. **Model Prediction Analysis**: Analyzes the model's predictions.
> 4. **Summary Report**: Generates a summary report based on the model performance and prediction analysis.

**Conditional Execution:**
> - Each task (except the initial deployment) includes conditions that depend on the success of the preceding tasks. If conditions are met, the next task in the Job is triggered.

**File Operations:**
> - The constructed JSON string is written to a file named `workflow-job-demo.json` in write mode, ensuring that the entire JOb configuration is saved externally for deployment purposes.

This setup is essential for automating model deployment workflows in a controlled and predictable manner, allowing for efficient scaling and maintenance of machine learning models.

In [0]:
# Define the configuration for the workflow
workflow_config_testing = f"""
{{
  "name": "{DA.username}-testing-workflow-job",
  "email_notifications": {{
    "on_failure": [
      "{DA.username}"
    ],
    "no_alert_for_skipped_runs": false
  }},
  "webhook_notifications": {{}},
  "notification_settings": {{
    "no_alert_for_skipped_runs": false,
    "no_alert_for_canceled_runs": false
  }},
  "timeout_seconds": 0,
  "max_concurrent_runs": 1,
  "tasks": [
    {{
      "task_key": "Deploy_best_model",
      "run_if": "ALL_SUCCESS",
      "notebook_task": {{
        "notebook_path": "{Deploy_model_notebook_path}",
        "source": "WORKSPACE"
      }},
      "existing_cluster_id": "{cluster_id}",
      "timeout_seconds": 0,
      "email_notifications": {{}},
      "notification_settings": {{
        "no_alert_for_skipped_runs": false,
        "no_alert_for_canceled_runs": false,
        "alert_on_last_attempt": false
      }},
      "webhook_notifications": {{}}
    }},
    {{
      "task_key": "Model_Performance_Test",
      "depends_on": [
        {{
          "task_key": "Deploy_best_model"
        }}
      ],
      "run_if": "ALL_SUCCESS",
      "notebook_task": {{
        "notebook_path": "{Parallel_Task_1_notebook_path}",
        "source": "WORKSPACE"
      }},
      "existing_cluster_id": "{cluster_id}",
      "timeout_seconds": 0,
      "email_notifications": {{}},
      "notification_settings": {{
        "no_alert_for_skipped_runs": false,
        "no_alert_for_canceled_runs": false,
        "alert_on_last_attempt": false
      }},
      "webhook_notifications": {{}}
    }},
    {{
      "task_key": "Model_Prediction_Analysis",
      "depends_on": [
        {{
          "task_key": "Deploy_best_model"
        }}
      ],
      "run_if": "ALL_SUCCESS",
      "notebook_task": {{
        "notebook_path": "{Parallel_Task_2_notebook_path}",
        "source": "WORKSPACE"
      }},
      "existing_cluster_id": "{cluster_id}",
      "timeout_seconds": 0,
      "email_notifications": {{}},
      "notification_settings": {{
        "no_alert_for_skipped_runs": false,
        "no_alert_for_canceled_runs": false,
        "alert_on_last_attempt": false
      }},
      "webhook_notifications": {{}}
    }},
    {{
      "task_key": "summary_report",
      "depends_on": [
        {{
          "task_key": "Model_Prediction_Analysis"
        }},
        {{
          "task_key": "Model_Performance_Test"
        }}
      ],
      "run_if": "ALL_SUCCESS",
      "notebook_task": {{
        "notebook_path": "{Summary_report_path}",
        "source": "WORKSPACE"
      }},
      "existing_cluster_id": "{cluster_id}",
      "timeout_seconds": 0,
      "email_notifications": {{}},
      "notification_settings": {{
        "no_alert_for_skipped_runs": false,
        "no_alert_for_canceled_runs": false,
        "alert_on_last_attempt": false
      }},
      "webhook_notifications": {{}}
    }}
  ],
  "queue": {{
    "enabled": true
  }},
  "run_as": {{
    "user_name": "{DA.username}"
  }}
}}
"""

# Write the configuration to a JSON file
with open('workflow-job-demo.json', 'w') as file:
    file.write(workflow_config_testing)

print("Workflow configuration file 'workflow-job-demo.json' created successfully.")

## Creating and Running a Lakeflow Job

This section details the process of creating and executing a Lakeflow job using the Databricks CLI:

1. **Creating the Job**:
   - The job is created by passing the JSON configuration file `workflow-job-demo.json` to the `databricks jobs create` command. This command returns a JSON object containing details of the created job, including the `job_id`.

2. **Extracting the Job ID**:
   - The `job_id` is extracted from the JSON output using a combination of `grep` and `awk`. The `grep` command isolates the line containing `job_id`, and `awk` is used to select the second field (the actual ID value), which is then stripped of extra characters using `tr`.

3. **Running the Job**:
   - With the `job_id` extracted, the job is initiated using `databricks jobs run-now`. This command triggers the execution of the Job defined in the job configuration file.

This process automates the deployment of tasks defined in the Databricks environment, ensuring that the model deployment and associated tasks are handled efficiently.

In [0]:
%sh
# Create the job and capture the output
output=$(databricks jobs create --json @workflow-job-demo.json)
echo $output

# Extract the job_id from the output
job_id=$(echo $output | grep -o '"job_id":[0-9]*' | awk -F':' '{print $2}')
echo "Extracted job_id: $job_id"

# Run the job using the extracted job_id
databricks jobs run-now $job_id

## Monitoring and Exploring the Executing Lakeflow Job

To effectively manage and gain insights from your executing Lakeflow job in the Databricks environment, follow these steps:

1. **Access the Jobs Console**:
   - From the Databricks sidebar, navigate to the **Job Runs** section, which lists all configured jobs.

2. **Find and View the Job**:
   - Use the `job_id` (`{{job_id}}`) or `{DA.username}-deploy-workflow-job` to locate your job. Click on the job name to access its details page.

3. **Explore Tasks**:
   - The job's details page displays its current status (e.g., *running*, *success*, *failure*). Click on a **Tasks tab** to view the created Job.  Click on each task to see the details.

4. **Explore Run Outputs**:
   - Go back to the **Runs Tab** click on the Runs.  Investigate the output logs, metrics, etc. of tasks for debugging information or to verify successful execution.

# Conclusion
In this demo, we have successfully set up and executed a Lakeflow job using the Databricks CLI. We configured the job to deploy a model, perform performance tests, analyze predictions, and generate a summary report. This process demonstrated how to automate and streamline model deployment and testing in Databricks, leveraging the CLI for efficient job management.

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>